# Module 2 — Deploy to AgentCore Runtime

In Module 1 your Chief of Staff agent ran on your laptop. Great for building — but no one else can reach
it. In this module you take the **exact same agent** and deploy it to **Amazon Bedrock AgentCore
Runtime**: a managed, serverless runtime that runs your agent in an isolated microVM and exposes it over
HTTP.

You'll deploy it **bare** — just the agent, in production. **No memory, no observability dashboards yet.**
That's deliberate: by the end you'll hit a real limitation (the agent forgets you between calls) that
**Module 3** fixes.

### What you'll do

| Step | What happens |
|------|--------------|
| **1. Recap the reuse** | See how the deployed agent reuses Module 1's `build_agent_options()` — one source of truth |
| **2. The entrypoint** | Understand the thin `@app.entrypoint` wrapper |
| **3. Configure** | Point AgentCore at your AWS account; review `agentcore.json` |
| **4. Test locally** | `agentcore dev` runs the container locally |
| **5. Deploy** | `agentcore deploy` builds the image, pushes to ECR, creates the runtime |
| **6. Invoke** | Call your live agent over HTTP and see the response |
| **7. Hit the wall** | Notice it's stateless — the hook into Module 3 |
| **8. Clean up** | Tear down so nothing keeps billing |

## Why AgentCore Runtime?

Getting an agent to production normally means building session isolation, scaling, credential management,
and an HTTP layer yourself. AgentCore Runtime gives you all of that with a single deploy:

| Feature | What you get |
|---|---|
| **Isolated microVM per session** | Each session runs in its own sandbox |
| **Serverless & auto-scaling** | No servers to manage; scales with demand |
| **Managed identity/credentials** | The runtime gets a scoped IAM role automatically |
| **HTTP protocol** | Invoke over HTTP; streaming supported |

You bring the agent; AgentCore runs it.

## Setup

Run the cell below to install all dependencies (Node.js, AgentCore CLI, Python packages) and
register the Jupyter kernel. After it completes, **select the `module-2-deploy` kernel** from the
kernel picker (top-right) and continue with the rest of the notebook.

In [1]:
!bash setup.sh

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
⠏npm warn deprecated glob@7.2.3: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 336 packages, and audited 374 packages in 8s
⠇
⠇46 packages are looking for funding
⠇  run `npm fund` for details
⠇
1 moderate severity vulnerability

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
⠇Resolved 164 packages in 0.58ms
Checked 142 packages in 0.98ms
Installed kernelspec module-2-deploy in /home/participant/.local/share/jupyter/kernels/module-2-

In [ ]:
!curl -fsSL https://rpm.nodesource.com/setup_20.x | sudo bash -
!sudo yum install -y nodejs-20.20.2
!sudo npm install -g @aws/agentcore@0.17.0
!cd agentcore/cdk && npm ci
!agentcore --version

2026-06-06 02:26:22 - Cleaning up old repositories...
2026-06-06 02:26:22 - Old repositories removed
2026-06-06 02:26:22 - Supported architecture: aarch64
2026-06-06 02:26:22 - Added N|Solid repository for LTS version: 20.x
2026-06-06 02:26:22 - dnf available, updating...
Node.js Packages for Linux RPM based distros -  147 kB/s | 3.0 kB     00:00    
Metadata cache created.
N|Solid Packages for Linux RPM based distros -  152 kB/s | 3.0 kB     00:00    
Metadata cache created.
2026-06-06 02:26:23 - Repository is configured and updated.
2026-06-06 02:26:23 - You can use N|solid Runtime as a node.js alternative
2026-06-06 02:26:23 - To install N|solid Runtime, run: dnf install nsolid -y
2026-06-06 02:26:23 - Run 'dnf install nodejs -y' to complete the installation.
Package nodejs-2:20.20.2-1nodesource.aarch64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹

### Install dependencies & register kernel

In [ ]:


import subprocess, shutil
from pathlib import Path

module_dir = Path.cwd()
module_name = module_dir.name  # e.g. "module-2-deploy"

# 1) .env
if not (module_dir / ".env").exists():
    shutil.copy(".env.example", ".env")
    print("✅ created .env from .env.example — edit it if you need a different model/region")
else:
    print("✅ .env already exists")

# 2) uv sync
subprocess.run(["uv", "sync"], check=True, cwd=module_dir)
print("✅ uv sync done")

# 3) Register Jupyter kernel
venv_python = module_dir / ".venv" / "bin" / "python"
subprocess.run([
    str(venv_python), "-m", "ipykernel", "install",
    "--user", "--name", module_name, "--display-name", module_name,
], check=True)
print(f"✅ Kernel registered: {module_name}")

✅ created .env from .env.example — edit it if you need a different model/region


Using CPython 3.11.14
Removed virtual environment at: .venv
Creating virtual environment at: .venv
Resolved 164 packages in 0.64ms
Installed 142 packages in 330ms
 + annotated-types==0.7.0
 + anyio==4.13.0
 + asgiref==3.11.1
 + asttokens==3.0.1
 + attrs==26.1.0
 + aws-opentelemetry-distro==0.17.1
 + bedrock-agentcore==1.13.0
 + boto3==1.43.21
 + botocore==1.43.24
 + cachetools==6.2.4
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + claude-agent-sdk==0.2.88
 + click==8.4.1
 + comm==0.2.3
 + cryptography==48.0.0
 + debugpy==1.8.21
 + decorator==5.3.1
 + executing==2.2.1
 + googleapis-common-protos==1.75.0
 + grpcio==1.81.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + idna==3.18
 + importlib-metadata==8.7.1
 + ipykernel==7.2.0
 + ipython==9.14.1
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 + jmespath==1.1.0
 + jsonschema==4.26.0
 + jsonschema-specifications==2025.9.1
 + jupyter-client==8.9.0
 + jupyter-core==5.9.1
 + matplotlib-inline

✅ uv sync done
Installed kernelspec module-2-deploy in /home/participant/.local/share/jupyter/kernels/module-2-deploy
✅ Kernel registered: module-2-deploy


In [16]:
#TODO
#Adding streenshot to show how to select the registerd kener name

## Step 1 — One agent, two front doors (the reuse)

We are **not** rewriting the agent. Module 1 and Module 2 share **one source of truth** for the agent's
identity — `build_agent_options()` in `agent.py`:

- **Module 1 (local):** `send_query()` calls `build_agent_options()` and runs the agent in-process.
- **Module 2 (deploy):** `agent_agentcore.py` calls the *same* `build_agent_options()` inside an AgentCore
  entrypoint.

Run the cell below to see that the deployment entrypoint contains **no agent logic of its own** — it just
wraps and reuses.

In [ ]:
import sys
sys.path.insert(0, "chief_of_staff_agent")

import inspect
from agent import build_agent_options
import agent_agentcore

# The entrypoint imports the shared identity builder...
src = inspect.getsource(agent_agentcore)
assert "from agent import build_agent_options" in src
assert "build_agent_options()" in src
# ...and does NOT redefine the system prompt or tool list.
assert "system_prompt" not in src.replace("build_agent_options", "")
print("✅ agent_agentcore.py reuses build_agent_options() — no duplicated agent logic")

opts = build_agent_options()
print("   tools:", opts.allowed_tools)
print("   setting_sources:", opts.setting_sources)

✅ agent_agentcore.py reuses build_agent_options() — no duplicated agent logic
   tools: ['Task', 'Read', 'Write', 'Edit', 'Bash', 'WebSearch']
   setting_sources: ['project']


## Step 2 — The AgentCore entrypoint

AgentCore runs your agent through a small **entrypoint**: a handler that receives a request payload and
streams a response. Here is the whole thing (`chief_of_staff_agent/agent_agentcore.py`):

```python
from bedrock_agentcore import BedrockAgentCoreApp
from claude_agent_sdk import ClaudeSDKClient
from agent import build_agent_options          # ← reuse Module 1's identity

app = BedrockAgentCoreApp()

@app.entrypoint
async def invoke(payload: dict):
    prompt = (payload or {}).get("prompt")
    options = build_agent_options()             # ← same config as local
    async with ClaudeSDKClient(options=options) as agent:
        await agent.query(prompt)
        async for msg in agent.receive_response():
            for block in getattr(msg, "content", []) or []:
                if getattr(block, "text", None):
                    yield block.text             # ← stream text back

if __name__ == "__main__":
    app.run()                                    # serves /invocations + /ping on :8080
```

`BedrockAgentCoreApp` implements the runtime's HTTP contract (`/invocations`, `/ping`) for you. The Module
1 agent logic moves inside this handler **unchanged**, because it's the same `build_agent_options()`.

## Why a Container build (not a zip)?

AgentCore supports two build types: **CodeZip** (Python zipped to S3) and **Container** (a Docker image).
We use **Container**, and here's the concrete reason:

> The Claude Agent SDK ships a ~218MB **native CLI binary**. Packaged as a zip, that binary arrives in the
> runtime **without its execute permission**, and the agent dies at startup with
> `Permission denied: .../claude_agent_sdk/_bundled/claude`.

A Container build `pip install`s the SDK **inside a Linux/ARM64 image**, so the binary has the right
architecture and permissions. The `Dockerfile` lives in `chief_of_staff_agent/`. (AgentCore Runtime
requires `linux/arm64` images.)

## Step 3 — Configure the deployment

The deployment is declared in `agentcore/agentcore.json` (already set up for you). Two things you must
personalize, because they depend on **your** AWS account:

1. **Deployment target** — your account ID + region (the next cell detects these automatically via the AWS CLI).

2. **Environment / model** — already set in `agentcore.json` `envVars` (Bedrock model IDs). Adjust if you
   use different models.

Run the cell below to generate `agentcore/aws-targets.json`, then validate the config:

In [ ]:
import json, os, subprocess

# --- Resolve region: single source of truth for the whole notebook ---
# Priority: AWS_REGION env (set by Workshop Studio) → AWS CLI config → fallback
_cli_region = subprocess.run(
    ["aws", "configure", "get", "region"], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get("AWS_REGION") or _cli_region or "us-west-2"

account_id = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
    capture_output=True, text=True,
).stdout.strip()

# Generate aws-targets.json from the resolved values
targets = [
    {
        "name": "default",
        "description": "Workshop deployment target (auto-generated).",
        "account": account_id,
        "region": REGION,
    }
]

with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)

print(f"✅ agentcore/aws-targets.json written:")
print(f"   account: {account_id}")
print(f"   region:  {REGION}")

In [ ]:
!agentcore validate

Valid


Let's look at what we're about to deploy — the runtime entry in `agentcore.json`:

In [ ]:
import json
cfg = json.load(open("agentcore/agentcore.json"))
print(json.dumps(cfg["runtimes"][0], indent=2))
# Note: build=Container, protocol=HTTP, enableOtel=true (traces → CloudWatch, explored in Module 4).
# The execution IAM role (Bedrock invoke + CloudWatch Logs + X-Ray) is created automatically by the CDK.

{
  "name": "cos",
  "build": "Container",
  "entrypoint": "agent_agentcore.py",
  "codeLocation": "chief_of_staff_agent/",
  "runtimeVersion": "PYTHON_3_14",
  "networkMode": "PUBLIC",
  "protocol": "HTTP",
  "instrumentation": {
    "enableOtel": true
  },
  "envVars": [
    {
      "name": "CLAUDE_CODE_USE_BEDROCK",
      "value": "1"
    },
    {
      "name": "ANTHROPIC_MODEL",
      "value": "global.anthropic.claude-opus-4-6-v1"
    },
    {
      "name": "ANTHROPIC_SMALL_FAST_MODEL",
      "value": "global.anthropic.claude-haiku-4-5-20251001-v1:0"
    }
  ]
}


## Step 4 — Test locally first (`agentcore dev`)

Before deploying to AWS, run the agent locally in a container that mimics the runtime. This is the fast
dev loop. In a **terminal** (it's a long-running server):

```bash
agentcore dev                     # builds + runs the container locally on :8080
```

Then, in another terminal, invoke it locally:

```bash
agentcore dev "What is our current monthly burn rate?"
```

You should see the Chief of Staff answer using the company data — exactly like Module 1, but now running
through the AgentCore HTTP contract.

In [ ]:
# this may take a while as it needs to build the container, please wait for
!agentcore dev --no-browser  

In [28]:
!agentcore dev --stream  "What is our current monthly burn rate?" -p 8081

Based on the company context in CLAUDE.md, our **monthly burn rate is approximately $500,000**.

However, I should note that this data appears to be from early 2024, and today's date is June 6, 2026. Let me check if we have more recent financial data available:Based on the available financial data, I can see we have burn rate information through **June 2024** (about 2 years ago from today's date). 

**As of June 2024:**
- **Monthly Burn Rate**: $525,000 (gross expenses)
- **Net Burn Rate**: $235,000 (after revenue)
- **Monthly Revenue**: $290,000
- **Headcount**: 53 employees

The burn rate increased from $450K in January 2024 to $525K by June 2024 as we scaled headcount from 45 to 53 people.

**However, this data is ~2 years old.** Would you like me to:
1. Use the **financial-analysis skill** to generate an updated projection based on growth trends?
2. Look for more recent financial records?
3. Run the financial forecasting scripts to estimate current burn rate?

Let me know how you'd

## Step 5 — Deploy to AgentCore Runtime

This builds the container image, pushes it to ECR, and provisions the runtime via CDK. It takes a few
minutes and creates real AWS resources. Run in a **terminal**:

```bash
agentcore deploy -y
```

When it finishes you'll get a runtime ARN. Check status anytime:

```bash
agentcore status
```

In [9]:

# this may take a while and please wait for the deployment complete
!agentcore deploy -y


✓ Load deployment target
⠋ Validate project...(node:287047) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
✓ Synthesize CloudFormation...
⠸ Check bootstrap status......(node:287047) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Check bootstrap status
✓ Check stack status...
✓ Deploy to AW

In [10]:
!agentcore status

(node:290634) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
AgentCore Status (target: default, us-west-2)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-west-2:3616919131
59:runtime/cosdeploy_cos-eSCPOSBXn5)
  URL: https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-west-2%3A361691913159%3Aruntime%2Fcosdeploy_cos-eSCPOSBXn5/
invocations


In [11]:
# You can also check status from the notebook once deployed:
import subprocess
out = subprocess.run(["agentcore", "status"], capture_output=True, text=True)
print(out.stdout or out.stderr)

AgentCore Status (target: default, us-west-2)

Agents
  cos: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-west-2:3616919131
59:runtime/cosdeploy_cos-eSCPOSBXn5)
  URL: https://bedrock-agentcore.us-west-2.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-west-2%3A361691913159%3Aruntime%2Fcosdeploy_cos-eSCPOSBXn5/
invocations



## Step 6 — Invoke your deployed agent

The agent is now live and reachable over HTTP. Invoke it:

In [13]:
!agentcore invoke --stream "What is our current runway and cash position?"

(node:292720) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
## Current Runway & Cash Position Summary

Here's a breakdown based on our financial data:

### Cash Position
| Metric | Value |
|--------|-------|
| **Starting Cash (Series A)** | $10,000,000 |
| **Monthly Gross Burn** | ~$500,000 - $525,000 |
| **Monthly Revenue** | ~$290,000 (as of latest data) |
| **Net Monthly Burn** | ~$235,000 (trending down ✅) |
| **Daily Burn Rate** | ~$16,667 |

### Runway Analysis
| Scenario | Runway |
|----------|--------|
| **Gross Runway** (no revenue) | 20 months |
| **Net Runway** (with current revenue) | Significantly longer (~42 

That response was produced by your agent **running in AWS**, using the same `CLAUDE.md` company context
and the `financial-analysis` skill you built in Module 1 — now served from managed infrastructure.

> **Observability note:** traces for this invocation are already flowing to CloudWatch (we turned on
> `enableOtel`). We'll *explore* them in **Module 4**.

## Step 7 — The limitation you'll hit (hook into Module 3)

Try invoking twice, where the second call depends on the first:

```bash
agentcore invoke "My name is Sarah and I'm the CEO."
agentcore invoke "What's my name?"
```

The agent **won't remember**. Each invocation is independent — a fresh, isolated session. That's what
"stateless" means, and it's the right default for an auto-scaling runtime. But real products need to
remember their users.

::: That's exactly what **Module 3 — Add AgentCore Memory** fixes. :::

## Step 8 — Clean up

To avoid ongoing charges, tear down the runtime when you're done. Run in a **terminal**:

```bash
agentcore remove agent --name cos     # remove from config
agentcore deploy -y                   # apply removal → destroys the runtime/stack
```

Confirm nothing lingers:

In [18]:
!agentcore remove agent --name cos  

{"success":true,"resourceType":"agent","resourceName":"cos","message":"Removed agent 'cos'","note":"Your agent app source code has not been modified. Deploy with `agentcore deploy` to apply your removal changes to AWS."}


In [19]:
!agentcore deploy -y   

✓ Load deployment target
✓ Validate project...
⠋ Validate AWS credentials...(node:308224) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate AWS credentials
✓ Build CDK project...
⠋ Synthesize CloudFormation...(node:308224) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Synthesize CloudFormation
✓ Check bootstrap status...


In [20]:
import subprocess
out = subprocess.run(["agentcore", "status"], capture_output=True, text=True)
print(out.stdout or out.stderr)

AgentCore Status (target: )
No resources match the given filters.



## Key takeaways

- AgentCore Runtime gives you isolated, auto-scaling, managed hosting with a single `agentcore deploy`.
- You deployed your **existing** agent by wrapping it in a thin entrypoint that **reuses
  `build_agent_options()`** — no duplicated agent logic.
- The Claude Agent SDK's bundled binary means **Container builds** (Linux/ARM64), not zip.
- The CDK auto-creates the **execution IAM role**; `enableOtel` already ships traces to CloudWatch.
- A bare deployment is **stateless** — which is exactly what **Module 3 (Memory)** addresses next.

## Next steps

Continue to **Module 3 — Add AgentCore Memory** to make your deployed agent remember users across calls.